# Notebook 02 — Cold-Chain Analysis: LSTM + Anomaly Detection

This notebook covers:
1. Sensor data exploration and visualisation
2. LSTM temperature forecaster training and evaluation
3. LSTM Autoencoder anomaly detection threshold calibration
4. Isolation Forest cross-channel anomaly detection
5. ANN risk scorer evaluation
6. End-to-end pipeline walkthrough on a sample excursion event

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

SENSOR_CSV = PROJECT_ROOT / 'datasets' / 'cold_chain' / 'sensor_logs.csv'
ZONE = 'pharma_vault'

print(f'Sensor data path: {SENSOR_CSV}')

## 1. Sensor Data Exploration

In [ ]:
if SENSOR_CSV.exists():
    df = pd.read_csv(SENSOR_CSV, parse_dates=['timestamp'])
    df_zone = df[df['zone_id'] == ZONE].sort_values('timestamp').reset_index(drop=True)
    
    print(f'Zone: {ZONE}')
    print(f'Records: {len(df_zone):,}')
    print(f'Period: {df_zone["timestamp"].min()} → {df_zone["timestamp"].max()}')
    print('\nDescriptive Statistics:')
    display(df_zone[['temp_outlet', 'temp_door_zone', 'temp_cargo_centre', 'humidity']].describe().round(3))
else:
    # Generate synthetic data for demonstration
    print('Sensor CSV not found — generating synthetic data for demo')
    rng = np.random.default_rng(42)
    n = 60 * 24 * 90  # 90 days at 1-min resolution
    t = pd.date_range('2024-01-01', periods=n, freq='1min')
    df_zone = pd.DataFrame({
        'timestamp': t,
        'zone_id': ZONE,
        'temp_outlet': 5.0 + rng.normal(0, 0.3, n),
        'temp_door_zone': 5.5 + rng.normal(0, 0.5, n),
        'temp_cargo_centre': 4.8 + rng.normal(0, 0.2, n),
        'humidity': 65.0 + rng.normal(0, 2, n),
        'door_open_flag': (rng.uniform(0, 1, n) < 0.02).astype(float),
        'compressor_state': (rng.uniform(0, 1, n) < 0.6).astype(float),
        'compressor_current': 5.2 + rng.normal(0, 0.5, n),
        'compressor_rpm': 2800 + rng.normal(0, 100, n),
    })
    # Inject a synthetic excursion at day 30
    excursion_start = 30 * 24 * 60
    excursion_end = excursion_start + 180
    df_zone.loc[excursion_start:excursion_end, 'temp_outlet'] += np.linspace(0, 6, excursion_end - excursion_start + 1)
    df_zone.loc[excursion_start:excursion_end, 'temp_door_zone'] += 2.0
    print(f'Synthetic data generated: {len(df_zone):,} rows')

In [ ]:
# Temperature time series plot (first 7 days)
sample = df_zone.head(7 * 24 * 60)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

ax1.plot(sample['timestamp'], sample['temp_outlet'], label='temp_outlet', lw=0.8)
ax1.plot(sample['timestamp'], sample['temp_door_zone'], label='temp_door_zone', lw=0.8, alpha=0.7)
ax1.plot(sample['timestamp'], sample['temp_cargo_centre'], label='temp_cargo_centre', lw=0.8, alpha=0.7)
ax1.axhline(2.0, color='blue', ls='--', lw=0.7, label='Min threshold')
ax1.axhline(8.0, color='red', ls='--', lw=0.7, label='Max threshold')
ax1.set_ylabel('Temperature (°C)')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_title(f'Cold-Chain Sensor Data — {ZONE} (7-day sample)')

ax2.plot(sample['timestamp'], sample['compressor_current'], color='orange', lw=0.6, label='Compressor current (A)')
ax2.plot(sample['timestamp'], sample['humidity'] / 10, color='teal', lw=0.6, label='Humidity / 10 (%RH)')
ax2.set_ylabel('Sensor value')
ax2.set_xlabel('Time')
ax2.legend(loc='upper right', fontsize=8)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. LSTM Forecaster Training

In [ ]:
import subprocess, shlex

# Save synthetic data for training script
train_csv = PROJECT_ROOT / 'datasets' / 'cold_chain' / 'sensor_logs.csv'
train_csv.parent.mkdir(parents=True, exist_ok=True)
if not train_csv.exists():
    df_zone.to_csv(train_csv, index=False)
    print(f'Saved training data → {train_csv}')

# Launch training (comment out if running interactively)
cmd = [
    sys.executable, str(PROJECT_ROOT / 'scripts' / 'train_lstm.py'),
    '--mode', 'forecaster',
    '--data', str(train_csv),
    '--output', str(PROJECT_ROOT / 'models' / 'weights' / 'lstm_forecaster.pt'),
    '--epochs', '20',  # Reduce for demo; use 100 for production
    '--batch', '64',
]
print('Training command:', ' '.join(cmd))
# result = subprocess.run(cmd, capture_output=True, text=True)
# print(result.stdout)
print('(Uncomment subprocess.run to actually run training)')

## 3. Anomaly Threshold Calibration

In [ ]:
# Reconstruct validation sequences with the trained autoencoder
SENSOR_COLS = [
    'temp_outlet', 'temp_door_zone', 'temp_cargo_centre',
    'humidity', 'door_open_flag', 'compressor_state',
    'compressor_current', 'compressor_rpm'
]
SEQ_LEN = 60

# Normalise
arr = df_zone[SENSOR_COLS].astype('float32').values
arr_norm = (arr - arr.min(axis=0)) / (arr.max(axis=0) - arr.min(axis=0) + 1e-8)

# Build sequences
seqs = np.array([arr_norm[i:i+SEQ_LEN] for i in range(len(arr_norm) - SEQ_LEN)], dtype='float32')
print(f'Total sequences: {len(seqs):,}')

# Simulate reconstruction errors with a random baseline
# (replace with actual model predictions when trained)
rng = np.random.default_rng(99)
recon_errors = rng.exponential(scale=0.005, size=len(seqs))
# Inject higher errors at excursion window
exc_idx = slice(30 * 24 * 60, 30 * 24 * 60 + 180)
recon_errors[exc_idx] *= 15.0

threshold_95 = np.percentile(recon_errors, 95)
print(f'95th percentile threshold: {threshold_95:.6f}')

plt.figure(figsize=(14, 4))
plt.plot(recon_errors, lw=0.5, color='steelblue', label='Reconstruction error')
plt.axhline(threshold_95, color='red', ls='--', label=f'95th pct threshold ({threshold_95:.4f})')
plt.xlabel('Sequence index')
plt.ylabel('MSE')
plt.title('LSTM Autoencoder Reconstruction Error Distribution')
plt.legend()
plt.yscale('log')
plt.tight_layout()
plt.show()

## 4. End-to-End Pipeline Walkthrough

In [ ]:
from src.cold_chain.pipeline import ColdChainPipeline, ZONE_THRESHOLDS
from src.cold_chain.lstm_forecaster import LSTMTemperatureForecaster
from src.cold_chain.anomaly_detector import LSTMAutoencoder, IsolationForestDetector
from src.cold_chain.ann_risk_scorer import ANNRiskScorer

# Build pipeline with untrained (lazy-loading) models
pipeline = ColdChainPipeline(
    zone_id=ZONE,
    lstm_forecaster=LSTMTemperatureForecaster('models/weights/lstm_forecaster.pt'),
    lstm_ae=LSTMAutoencoder('models/weights/lstm_ae.pt'),
    isolation_forest=IsolationForestDetector('models/weights/iforest.joblib'),
    ann_scorer=ANNRiskScorer('models/weights/ann_risk.h5'),
)

# Use a 60-row window from the synthetic excursion period
excursion_window = arr_norm[30 * 24 * 60: 30 * 24 * 60 + 60]
latest_5 = excursion_window[-1, [0, 3, 5, 6, 7]]

try:
    result = pipeline.evaluate(excursion_window, latest_5)
    print(f'Zone: {result.zone_id}')
    print(f'Risk level: {result.overall_risk_level}')
    print(f'Requires immediate action: {result.requires_immediate_action}')
    print(f'Alerts: {len(result.alerts)}')
    for alert in result.alerts:
        print(f'  [{alert.alert_level.upper()}] {alert.source}')
except Exception as e:
    print(f'Pipeline evaluation (models not loaded): {e}')
    print('Train models first using scripts/train_lstm.py and scripts/train_ann.py')